# Notebook 02 — Exploratory Data Analysis (EDA)

**Project:** Credit Portfolio Segmentation and Risk Pattern Analysis
**Covers:** Stage 3 of the project document — *analyse feature distributions, correlations, borrower characteristics, and default patterns.*

**What this notebook does**
1. **Distributions** — what typical borrowers and loans look like (income, debt-to-income, loan amount, FICO, purpose, home ownership, employment).
2. **Correlations** — whether the numeric features repeat each other or carry different information.
3. **Default patterns** — how the historical default rate changes with each characteristic, across US states and over time.
4. Saves 7 charts to `reports/figures/` and every default-rate table to `reports/eda_default_rates.csv`. Together they form the **EDA report** deliverable.

**Before you run it:** notebook 01 must have been run once, because this notebook reads `data/processed/loans_clean.pkl`. Select the `.venv` kernel, then click **Run All** (about 1 minute).

> **Golden rule:** here we *describe* historical defaults. `Default` is never an input to the clustering in Stages 4–6.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make `src` importable (this notebook runs from the notebooks/ folder).
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CLEAN_DATA_FILE, FIGURES_DIR, REPORTS_DIR, TARGET_COLUMN
from src.eda import (
    add_value_bands, default_rate_table, plot_correlation_heatmap,
    plot_default_rates,
)
from src.plot_style import (
    COMPACT_FORMATTER, DOLLAR_FORMATTER, PRIMARY_COLOR, SURFACE,
    TEXT_PRIMARY, TEXT_SECONDARY, apply_style,
)

apply_style()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not CLEAN_DATA_FILE.exists():
    raise FileNotFoundError(
        f"Run notebook 01 first: it creates {CLEAN_DATA_FILE}"
    )

loans = add_value_bands(pd.read_pickle(CLEAN_DATA_FILE))
OVERALL_RATE = loans[TARGET_COLUMN].mean() * 100
print(f"{len(loans):,} loans | overall default rate {OVERALL_RATE:.2f}%")

`add_value_bands()` (in `src/eda.py`) adds four **banded** columns — `fico_band`, `dti_band`, `income_band`, `loan_band` — that group a continuous value into ranges such as FICO "700–719". Bands make default patterns easy to read: one default rate per range instead of one per exact value.

---
## 1. What do borrowers and loans look like?

### 1.1 Numeric features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
(ax_income, ax_dti), (ax_loan, ax_fico) = axes

income = loans["revenue"]
log_bins = np.logspace(np.log10(income.min()), np.log10(income.max()), 60)
ax_income.hist(income, bins=log_bins, color=PRIMARY_COLOR, rwidth=0.9)
ax_income.set_xscale("log")
ax_income.xaxis.set_major_formatter(DOLLAR_FORMATTER)
ax_income.set_title("Annual income (log scale)")
ax_income.set_xlabel("Annual income (USD)")

DTI_AXIS_LIMIT = 60
dti = loans["dti_n"].dropna()
ax_dti.hist(dti[dti <= DTI_AXIS_LIMIT], bins=60, color=PRIMARY_COLOR,
            rwidth=0.9)
ax_dti.set_title("Debt-to-income ratio")
ax_dti.set_xlabel("DTI (%)")
ax_dti.text(0.97, 0.95, f"{(dti > DTI_AXIS_LIMIT).sum():,} loans above "
            f"{DTI_AXIS_LIMIT}% not shown", transform=ax_dti.transAxes,
            ha="right", va="top", fontsize=9, color=TEXT_SECONDARY)

ax_loan.hist(loans["loan_amnt"], bins=np.arange(0, 41_000, 1_000),
             color=PRIMARY_COLOR, rwidth=0.9)
ax_loan.xaxis.set_major_formatter(DOLLAR_FORMATTER)
ax_loan.set_title("Loan amount ($1,000 bins)")
ax_loan.set_xlabel("Loan amount (USD)")

ax_fico.hist(loans["fico_n"], bins=np.arange(610, 851, 5),
             color=PRIMARY_COLOR, rwidth=0.9)
ax_fico.set_title("FICO credit score")
ax_fico.set_xlabel("FICO score")

for ax in axes.flat:
    ax.set_ylabel("Number of loans")
    ax.yaxis.set_major_formatter(COMPACT_FORMATTER)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_numeric_distributions.png")
plt.show()

**How to read the charts.**
- **Income** (log scale): a single hump centred near \$65,000. Most borrowers earn \$30K–\$150K. The jagged top comes from people reporting **round numbers**: 48% of incomes are exact multiples of \$5,000.
- **DTI**: a bell-like shape centred around 17–18%, with fewer loans above 35–40%. Lending Club rarely approved very high DTIs.
- **Loan amount**: tall spikes at \$10K, \$12K, \$15K, \$20K and \$35K. **66% of loans are whole thousands** — borrowers ask for round amounts.
- **FICO**: piled up at the low end (660–700) and thinning out above 750. The platform approved few scores below about 660, so the left edge is cut off sharply.

**Why it matters for clustering:** income is skewed (handled by the log transform), DTI has a long right tail (handled by capping) and FICO sits in a narrow band. All four will be **standardised** in Stage 4 so that no feature dominates the distances just because of its units.

### 1.2 Categorical features

In [ ]:
fig, (ax_purpose, ax_home, ax_emp) = plt.subplots(
    1, 3, figsize=(15, 5), gridspec_kw={"width_ratios": [1.3, 1, 1]}
)

purpose_counts = loans["purpose"].value_counts(ascending=True)
ax_purpose.barh(purpose_counts.index.astype(str), purpose_counts,
                height=0.6, color=PRIMARY_COLOR)
ax_purpose.set_title("Loan purpose")

home_counts = loans["home_ownership_n"].value_counts(ascending=True)
ax_home.barh(home_counts.index.astype(str), home_counts, height=0.6,
             color=PRIMARY_COLOR)
ax_home.set_title("Home ownership")

emp_counts = loans["emp_length"].value_counts(sort=False).iloc[::-1]
ax_emp.barh(emp_counts.index.astype(str), emp_counts, height=0.6,
            color=PRIMARY_COLOR)
ax_emp.set_title("Employment length")

for ax in (ax_purpose, ax_home, ax_emp):
    ax.set_xlabel("Number of loans")
    ax.xaxis.set_major_formatter(COMPACT_FORMATTER)
    ax.grid(axis="y", visible=False)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_categorical_distributions.png")
plt.show()

**How to read the charts.**
- **Purpose**: debt consolidation (58%) and credit card refinancing (22%) dominate — **80% of loans are used to repay other debt**. The other 12 purposes together make up the last 20%. In Stage 4 we group the small ones (home, small business, personal/other), so that a category with a few hundred loans cannot create its own segment.
- **Home ownership**: mortgage holders (49%) and renters (40%) are the two big groups; OTHER is almost empty (517 loans).
- **Employment length**: "10+ years" is by far the largest group (33%), and **NI** (no information) is 5.8%.

---
## 2. Correlations between the numeric features

**Concept.** A correlation coefficient measures how two features move together: **+1** means they rise together perfectly, **−1** means one falls as the other rises, and **0** means no relationship. We use **Spearman** correlation, which works on ranks, because income and DTI are skewed and contain extreme values.

**Why it matters.** If two features were almost perfectly correlated, they would carry the same information and count twice in the clustering distances. PCA in Stage 5 is the tool that removes this kind of overlap.

In [ ]:
numeric_view = pd.DataFrame({
    "Income (log)": np.log1p(loans["revenue"]),
    "DTI": loans["dti_n"],
    "Loan amount": loans["loan_amnt"],
    "FICO": loans["fico_n"],
    "Employment years": loans["emp_length_years"],
})
correlations = numeric_view.corr(method="spearman")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
image = plot_correlation_heatmap(correlations, ax,
                                 "Spearman correlation of numeric features")
fig.colorbar(image, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_correlation_heatmap.png")
plt.show()
correlations.round(2)

**How to read the heatmap.** Red cells are positive correlations, blue cells are negative, and grey is close to zero.
- **Income ↔ loan amount: +0.49** — the only moderate link. Higher earners borrow more.
- **Income ↔ DTI: −0.19** — higher earners have slightly lower debt-to-income.
- Every other pair is close to 0 (|ρ| ≤ 0.14). **FICO is almost unrelated to income, DTI and loan size.**

**Conclusion:** the five features mostly carry **different** information, which is good for segmentation because each one adds something. It also explains why PCA in Stage 5 needs several components to keep most of the information.

---
## 3. Default patterns

The **default rate** of a group is the share of its loans that defaulted. The vertical line on each chart marks the **portfolio average (19.98%)**: bars to the right of it are riskier than the average loan, and bars to the left are safer. `lift` in the tables is the group's rate divided by the average: 1.5 means "50% more likely to default".

### 3.1 Credit and financial profile

In [ ]:
band_tables = {
    "FICO score": default_rate_table(loans, "fico_band"),
    "DTI (%)": default_rate_table(loans, "dti_band"),
    "Annual income": default_rate_table(loans, "income_band"),
    "Loan amount": default_rate_table(loans, "loan_band"),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, table) in zip(axes.flat, band_tables.items()):
    plot_default_rates(table, f"Default rate by {name}", ax, OVERALL_RATE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_default_rate_by_bands.png")
plt.show()

for name, table in band_tables.items():
    print(f"\n{name}")
    print(table.to_string())

**How to read the charts — four clear, one-directional patterns.**
- **FICO is the strongest single signal:** default falls from **25.3%** (below 680) to **8.3%** (760+). The best-scored borrowers default **3× less often** than the lowest-scored.
- **DTI:** default climbs from **14.8%** (DTI under 10) to **29–31%** (DTI 30+). The more of their income borrowers already spend on debt, the riskier they are.
- **Income:** default falls from **23.6%** (under \$40K) to **15.1%** (\$150K+).
- **Loan amount:** bigger loans default more — **15.5%** under \$5K versus **24.2%** at \$30K+.

These patterns are what we hope to see *again* across the segments in Stage 8: if the clusters capture real credit profiles, their default rates should differ, even though `Default` was never used to build them.

### 3.2 Purpose, home ownership and employment

In [ ]:
category_tables = {
    "purpose": default_rate_table(loans, "purpose"),
    "home ownership": default_rate_table(loans, "home_ownership_n"),
    "employment length": default_rate_table(loans, "emp_length"),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5),
                         gridspec_kw={"width_ratios": [1.3, 1, 1]})
plot_default_rates(category_tables["purpose"], "Default rate by purpose",
                   axes[0], OVERALL_RATE, sort=True)
plot_default_rates(category_tables["home ownership"],
                   "Default rate by home ownership", axes[1], OVERALL_RATE,
                   sort=True)
plot_default_rates(category_tables["employment length"],
                   "Default rate by employment length", axes[2], OVERALL_RATE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_default_rate_by_category.png")
plt.show()

category_tables["purpose"].sort_values("default_rate", ascending=False)

**How to read the charts.**
- **Purpose:** **small business loans are the riskiest (29.9%)**, followed by renewable energy and moving (about 23–24%). **Weddings (12.4%), cars (14.7%) and credit cards (16.9%)** are the safest. Small groups such as educational (423 loans) give less reliable rates.
- **Home ownership:** **renters default most (23.2%)**, mortgage holders least (17.2%), and outright owners sit in between (20.6%).
- **Employment length:** all known lengths sit close to average (18.8%–20.6%), but **NI — employment not reported — defaults at 27.0%**. That is why notebook 01 kept a 0/1 flag instead of quietly filling these values in: *not reporting* employment is itself informative.

---
## 4. Geographic portfolio analysis

We compare states with **at least 1,000 loans**. Smaller groups give unstable rates, where a few defaults swing the percentage.

In [ ]:
state_table = default_rate_table(loans, "addr_state", min_loans=1_000)
top_volume = state_table.sort_values("loans").tail(10)
extremes = pd.concat([
    state_table.nsmallest(10, "default_rate"),
    state_table.nlargest(10, "default_rate"),
])

fig, (ax_volume, ax_rate) = plt.subplots(1, 2, figsize=(13, 6.5))
ax_volume.barh(top_volume.index.astype(str), top_volume["loans"], height=0.6,
               color=PRIMARY_COLOR)
ax_volume.set_title("Top 10 states by number of loans", pad=16)
ax_volume.set_xlabel("Number of loans")
ax_volume.xaxis.set_major_formatter(COMPACT_FORMATTER)
ax_volume.grid(axis="y", visible=False)

plot_default_rates(extremes, "10 safest and 10 riskiest states", ax_rate,
                   OVERALL_RATE, sort=True)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_geography.png")
plt.show()

print(f"States compared: {len(state_table)}")
print("Lowest default rates:",
      state_table["default_rate"].nsmallest(5).to_dict())
print("Highest default rates:",
      state_table["default_rate"].nlargest(5).to_dict())

**How to read the charts.**
- **Volume:** the portfolio is concentrated in the largest states — **California (≈197K loans), Texas and New York (≈110K each) and Florida (≈96K)**.
- **Risk:** default rates range from about **13% (DC, Maine, Vermont)** to about **24–26% (Arkansas, Nebraska, Mississippi)**. A loan in the riskiest state is roughly twice as likely to default as one in the safest.

State is kept **out of the clustering**: 51 one-hot columns would swamp the financial features, and segmenting people by where they live is not the goal. Instead we will profile each segment by state afterwards.

---
## 5. Portfolio trend over time

Two separate charts (one measure each) — never two different scales on one chart.

In [ ]:
year_table = default_rate_table(loans, "issue_year")

fig, (ax_count, ax_trend) = plt.subplots(1, 2, figsize=(13, 4.5))
ax_count.bar(year_table.index.astype(str), year_table["loans"], width=0.6,
             color=PRIMARY_COLOR)
ax_count.set_title("Loans issued per year")
ax_count.set_ylabel("Number of loans")
ax_count.yaxis.set_major_formatter(COMPACT_FORMATTER)
ax_count.grid(axis="x", visible=False)

ax_trend.plot(year_table.index.astype(str), year_table["default_rate"],
              color=PRIMARY_COLOR, marker="o", markersize=7,
              markeredgecolor=SURFACE, markeredgewidth=2)
ax_trend.axhline(OVERALL_RATE, color=TEXT_PRIMARY, linewidth=1)
# Label placed over 2009-2011, where the line sits well below the average.
ax_trend.text(2, OVERALL_RATE + 0.4, f"Portfolio {OVERALL_RATE:.1f}%",
              fontsize=8, color=TEXT_SECONDARY, va="bottom")
ax_trend.set_title("Default rate by year issued")
ax_trend.set_ylabel("Default rate (%)")
ax_trend.set_ylim(0, year_table["default_rate"].max() * 1.2)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_trend_by_year.png")
plt.show()
year_table

**How to read the charts.**
- **Volume** grew fast from 2012 to a peak in **2015 (≈376K loans)**, then fell. As notebook 01 explained, the fall after 2016 mostly reflects how the dataset was built: only **finished** loans are included, and many 2017–2018 loans were still running.
- **Default rate** was lowest for **2009–2013 loans (≈14–16%)**, rose to **≈23% for 2016–2017**, then dropped to 15.8% for 2018.

⚠️ **Limitation — be careful with recent years.** For 2017–2018, only loans that had *already finished* are in the data. Early defaulters and early repayers finish quickest, so these years are a biased sample. Year-to-year comparisons are reliable up to about 2015. This is a good point to mention in the viva.

---
## 6. Save the EDA tables

In [ ]:
all_tables = {**band_tables, **category_tables,
              "state": state_table, "issue year": year_table}
eda_rates = pd.concat(
    {name: table.reset_index(names="group")
     for name, table in all_tables.items()},
    names=["feature"],
).reset_index(level="feature").reset_index(drop=True)

eda_rates_file = REPORTS_DIR / "eda_default_rates.csv"
eda_rates.to_csv(eda_rates_file, index=False)

spread = (eda_rates.groupby("feature")["default_rate"]
          .agg(lowest="min", highest="max"))
spread["spread_points"] = spread["highest"] - spread["lowest"]
print(f"Saved {len(eda_rates)} rows to {eda_rates_file.name}\n")
spread.sort_values("spread_points", ascending=False)

**How to read the output.** `spread_points` is the gap, in percentage points, between a feature's riskiest and safest group: a rough measure of how strongly the feature separates risk on its own. **Purpose has the widest spread (17.5 points)**, but partly because some of its groups are small (weddings, small business). **FICO (17.0) and DTI (15.9)** come right behind, and every one of their groups is large, which makes them the most reliable risk signals. Home ownership separates risk the least (6.0 points).

---
## EDA summary — key insights

| # | Insight | Evidence |
|---|---|---|
| 1 | Typical borrower: \$65K income, DTI ≈ 18%, FICO 660–720, borrowing \$10K–\$20K | Section 1 |
| 2 | 80% of loans refinance other debt (debt consolidation + credit card) | 1.2 |
| 3 | Features carry different information (only income ↔ loan amount is moderately correlated) | 2 |
| 4 | **Credit score is the clearest risk signal:** 25.3% → 8.3% default from the lowest to the highest FICO band | 3.1 |
| 5 | Risk rises with DTI and loan size, and falls with income | 3.1 |
| 6 | Small-business loans (29.9%), renters (23.2%) and borrowers without employment information (27.0%) are the riskiest groups | 3.2 |
| 7 | Default rates vary about twofold across states (≈13% to ≈26%) | 4 |
| 8 | Recent years (2017–2018) are a biased sample because only finished loans are included | 5 |

**What this means for Stage 4 (feature engineering):**
- Log-transform income; cap DTI's long tail; standardise all numeric features.
- Group the 14 purposes into 5 groups (debt consolidation, credit card, home, small business, personal/other).
- Keep the employment "unknown" flag.
- Keep state and issue year **out** of the clustering, and use them to profile the segments afterwards.

**Next:** Notebook 03 — feature engineering, scaling and dimensionality reduction (Stages 4–5).

---
## Viva practice — Stage 3

**1. What is EDA and why did you do it before clustering?**
Exploratory Data Analysis means summarising and visualising the data to understand its distributions, relationships and oddities before modelling. It told us which features need transforming (income, DTI), which to group (purpose) and which to keep out of the clustering (state, year).

**2. Which feature is most strongly related to default?**
FICO score: the default rate falls steadily from 25.3% below 680 to 8.3% at 760+.

**3. You used Default in the EDA — isn't that leakage?**
No. Leakage would mean using `Default` to *build* the segments. In the EDA we only describe historical outcomes. The clustering features in Stage 4 never include `Default`.

**4. Why Spearman instead of Pearson correlation?**
Spearman works on ranks, so it is robust to skewed data and extreme values such as the \$11M income. Pearson measures only straight-line relationships and is sensitive to outliers.

**5. Why not compare default rates for every state?**
States with few loans give unstable rates, so we only compared states with at least 1,000 loans.

**6. What is the main limitation you found?**
The dataset only contains finished loans, so recent years (2017–2018) are under-represented and biased toward loans that ended early.

**7. Why is state not a clustering feature?**
It would need 51 one-hot columns that would dominate the distances, and the goal is to group borrowers by financial and credit profile, not by location. State is used afterwards, to profile the segments.